# DBpedia — construção offline com integridade e aliases

O construtor usa somente a biblioteca padrão e está em `tools/dbpedia_builder.py`.
Não há instalação automática nem exclusão de cache. Esta execução consulta a
internet e pertence à preparação offline, fora do ambiente de avaliação.

A configuração preserva as sementes e filtros anteriores. O cache antigo não é
reutilizado: respostas sem estado de completude não são confiáveis. Para invalidar
explicitamente o novo cache, altere `cache_version` no JSON; para um snapshot fixo,
configure `endpoint` e `snapshot`. Um endpoint vivo não garante reprodução entre datas.


In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "tools/dbpedia_builder.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "tools/dbpedia_builder.py").is_file():
    raise RuntimeError("Execute a partir da raiz do repositório ou de notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tools.dbpedia_builder import build, validate_bundle

CONFIG = json.loads((ROOT / "notebooks/dbpedia-config.json").read_text())
CONFIG["redirect_review"] = json.loads((ROOT / "notebooks/dbpedia-redirect-review.json").read_text())
CACHE = ROOT / ".cache/dbpedia/extraction-v3.sqlite"
OUTPUT = ROOT / "experiments/runs/dbpedia-integrity-aliases-v2"
CACHE.parent.mkdir(parents=True, exist_ok=True)


## Aliases e limites

`reviewed_aliases` aceita registros com `text`, `canonical_uri`, `source` e
`language`. Inclua apenas associações globais revisadas com fonte. Não se presume
que DONES seja sinônimo de IFMIF. Siglas definidas nos artigos devem permanecer
locais ao documento com IDs de passagem; essa ligação não é feita nesta etapa.

Redirects e labels são preservados em `names`, sem corte de 15 aliases ou 35
caracteres. `alias_index.json` preserva todos os candidatos ambíguos. O consumidor
atual continua recebendo `aliases` como lista de strings.

A coleta pagina todos os tipos de consulta com ordenação e divide lotes com falha.
Falha de consulta ou ciclo de redirects interrompem a publicação. Sementes sem
label também interrompem, salvo exceções explícitas e justificadas em
`allowed_missing_seed_labels`, sempre registradas no manifesto. A configuração
exclui apenas `Region_of_practical_equivalence`, ausente na coleta relatada.
Uma ausência de metadados de não sementes é inventariada no manifesto. O tamanho
máximo agora se aplica ao diretório inteiro; não há poda automática. Se exceder,
revise o escopo e execute em um novo diretório. O limite não valida o ZIP de 1 GB.

A revisão em `dbpedia-redirect-review.json` rejeita equivalências específicas
e exclui o destino misto Demo. O manifesto conserva o caminho observado e o
motivo da revisão. O cache SPARQL continua reutilizável. Para revisar o bundle
v1 já coletado sem rede, usar `tools/revise_dbpedia_bundle.py`; ver o relatório
de decisão de aliases de 12/11/2026.


In [2]:
# Pode levar horas. Nunca sobrescreve OUTPUT nem os artefatos atuais em data/.
# Para outra execução completa, escolha um OUTPUT novo; o cache é reaproveitado.
if OUTPUT.exists():
    raise FileExistsError(f"Escolha um diretório novo: {OUTPUT}")
bundle = build(CONFIG, CACHE, OUTPUT)
manifest = validate_bundle(bundle)
print(json.dumps(manifest["counts"], indent=2))
print(f"Artefatos publicados: {bundle}")


{
  "cards": 8343,
  "edges": 60021,
  "triples": 79414
}
Artefatos publicados: /home/arthur/repos/IEEE-RAG-Challenge/experiments/runs/dbpedia-integrity-aliases-v1


## Saída e consumo

O diretório só aparece após validação, via rename do diretório de preparação.
`manifest.json` marca `complete`, guarda configuração, resolução das sementes,
URIs sem metadados, tamanhos e hashes. `domain_kg.ttl` usa N-Triples, subconjunto
válido e determinístico de Turtle. `edges.jsonl` preserva wikilinks por URI;
`kg_rag_chunks.jsonl` mantém os campos anteriores e adiciona nomes com procedência.
Falhas geram um arquivo separado `<OUTPUT>.failed.json`; nunca são um bundle válido.

O consumidor precisa selecionar explicitamente o novo caminho e verificar o
manifesto antes de experimentar. Nenhum recurso foi adicionado à submissão.
Avaliar aliases antes de reintroduzir expansão de uma aresta; manter o híbrido
como controle e medir o pipeline final na H100.
